# Finalização

Fecha a extração: restaura o registro de falhas, confere os documentos recuperados na última
rodada, reavalia os que haviam passado por critério afrouxado e regrava a tabela final.

**Contexto.** A extração passou por três tratamentos das falhas de localização. O primeiro
afrouxou os critérios de aceitação e recuperou 271 documentos. O segundo corrigiu o detector de
notas explicativas — que reprovava trechos legítimos, como o da Texas Instruments, por não
encontrar um rótulo específico — e recuperou mais 106, agora com critérios normais. Este
notebook faz o terceiro passo: verificar se os 271 do primeiro tratamento também passam pelos
critérios normais, agora que a detecção está correta. Os que passarem perdem a ressalva.

**Ordem.** De cima para baixo, sem pular. As seções 1 e 2 preparam o ambiente e o motor; as
seções 3 a 6 executam o fechamento.

## 1. Ambiente

In [ ]:
SEU_NOME  = "Helena Ribeiro"
SEU_EMAIL = "helenafariasr@gmail.com"

USAR_DRIVE = True
PASTA      = "/content/drive/MyDrive/SEC_PAINEL"

FORMULARIOS = ["10-K", "20-F", "40-F"]
SALVAR_HTML_DA_SECAO = False

REQ_POR_SEGUNDO = 8
N_THREADS       = 4
MAX_TENTATIVAS  = 4

In [ ]:
import os

if USAR_DRIVE:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive", force_remount=True)
        print("Drive montado.")
    except Exception as e:
        print("Não foi possível montar o Drive:", type(e).__name__, e)
        USAR_DRIVE = False
        PASTA = PASTA.replace("/content/drive/MyDrive", "/content")

for sub in ["", "indice", "pdf", "html_secao", "log"]:
    os.makedirs(os.path.join(PASTA, sub), exist_ok=True)
print("Pasta de trabalho:", PASTA)

!apt-get -qq update > /dev/null && apt-get -qq install -y wkhtmltopdf poppler-utils > /dev/null
!wkhtmltopdf --version | head -1 && pdftotext -v 2>&1 | head -1

In [ ]:
import io, os, re, json, time, random, threading, subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urljoin

import requests
import pandas as pd

assert "@" in SEU_EMAIL and SEU_NOME.strip(), "Preencha SEU_NOME e SEU_EMAIL."

HEADERS = {"User-Agent": f"{SEU_NOME} {SEU_EMAIL}", "Accept-Encoding": "gzip, deflate"}

_lock, _ultimo = threading.Lock(), [0.0]

def _esperar_vez():
    with _lock:
        agora = time.time()
        espera = _ultimo[0] + 1.0 / REQ_POR_SEGUNDO - agora
        if espera > 0:
            time.sleep(espera)
            agora = time.time()
        _ultimo[0] = agora

_local = threading.local()

def _sessao():
    if not hasattr(_local, "s"):
        s = requests.Session(); s.headers.update(HEADERS); _local.s = s
    return _local.s

def baixar(url, binario=False):
    """GET com limite de taxa, repetição e espera crescente. None se falhar."""
    for i in range(MAX_TENTATIVAS):
        _esperar_vez()
        try:
            r = _sessao().get(url, timeout=90)
            if r.status_code == 200:
                return r.content if binario else r.text
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep((2 ** i) + random.random())
    return None

print("Identificação enviada à SEC:", HEADERS["User-Agent"])

## 2. Motor de recorte e detectores corrigidos

In [ ]:
IGNORAR = re.compile(r"(?i)^(r\d+\.htm|report\d*\.htm|.*-index.*\.html?|"
                     r"\d{10}-\d{2}-\d{6}\.txt|.*_?cal\.|.*_?def\.|.*_?lab\.|.*_?pre\.|"
                     r".*\.xml|.*\.xsd|.*\.jpg|.*\.png|.*\.gif)$")
EXTENSOES = (".htm", ".html", ".txt", ".pdf")

def arquivos_do_protocolo(url_pasta, excluir=()):
    """Documentos do protocolo, do maior para o menor, incluindo os anexos em PDF."""
    txt = baixar(urljoin(url_pasta, "index.json"))
    if not txt:
        return []
    try:
        itens = json.loads(txt)["directory"]["item"]
    except (ValueError, KeyError):
        return []
    saida = []
    for i in itens:
        nome = i.get("name", "")
        if not nome.lower().endswith(EXTENSOES):
            continue
        if IGNORAR.match(nome) or nome in excluir:
            continue
        saida.append((nome, int(i.get("size") or 0)))
    saida.sort(key=lambda x: x[1], reverse=True)
    return [n for n, _ in saida]

P_DOCUMENTO = re.compile(r"(?is)<DOCUMENT>.*?<TYPE>([^\r\n<]*).*?<FILENAME>([^\r\n<]*)"
                         r".*?<TEXT>(.*?)</TEXT>")

def documentos_da_submissao(url_submissao):
    """Último recurso: a submissão completa reúne todos os documentos do protocolo."""
    bruto = baixar(url_submissao)
    if not bruto:
        return []
    saida = []
    for tipo, nome, corpo in P_DOCUMENTO.findall(bruto):
        nome = nome.strip()
        if not nome.lower().endswith((".htm", ".html", ".txt")):
            continue
        if len(corpo) > 40000:
            saida.append((nome, corpo))
    saida.sort(key=lambda x: len(x[1]), reverse=True)
    return saida[:6]

def documento_principal(linha):
    """O documento principal vem da tabela do painel; a listagem da pasta é a alternativa."""
    doc = linha.get("documento_principal")
    if not (isinstance(doc, str) and doc):
        nomes = arquivos_do_protocolo(linha["url_pasta"])
        doc = nomes[0] if nomes else None
    sic = str(linha.get("sic") or "")
    return (linha["url_pasta"] + doc if doc else None), doc, sic

In [ ]:
ENTIDADES = re.compile(r"&nbsp;|&#160;|&#xa0;|&#xA0;")
TAGS      = re.compile(r"(?is)<(script|style)\b.*?</\1>|<[^>]+>")

def texto_e_mapa(html):
    """Texto visível em minúsculas, espaços normalizados, com mapa texto -> posição no HTML."""
    saida, mapa, ultimo, espaco = [], [], 0, True
    def empurrar(ch, pos):
        nonlocal espaco
        if ch.isspace():
            if espaco:
                return
            saida.append(" "); mapa.append(pos); espaco = True
        else:
            saida.append(ch.lower()); mapa.append(pos); espaco = False
    for m in TAGS.finditer(html):
        for k, ch in enumerate(html[ultimo:m.start()]):
            empurrar(ch, ultimo + k)
        empurrar(" ", m.start())
        ultimo = m.end()
    for k, ch in enumerate(html[ultimo:]):
        empurrar(ch, ultimo + k)
    return "".join(saida), mapa

# Títulos de início e de fim
P_10K_INI = re.compile(r"item\s*8[\.\:\)\-—\s]{0,6}financial\s+statements")
P_10K_FIM = re.compile(r"item\s*9[a-c]?[\.\:\)\-—\s]{0,6}(changes\s+in\s+and|controls\s+and\s+procedures|other\s+information)")
P_20F_INI = re.compile(r"item\s*18[\.\:\)\-—\s]{0,6}financial\s+statements")
P_20F_ALT = re.compile(r"item\s*17[\.\:\)\-—\s]{0,6}financial\s+statements")
P_20F_FIM = re.compile(r"item\s*19[\.\:\)\-—\s]{0,6}exhibit")
P_AUDITOR = re.compile(r"report\s+of\s+independent|independent\s+auditor'?s?\s+report|"
                       r"auditors?'?\s+report\s+to|report\s+of\s+the\s+independent")
P_ASSINAT = re.compile(r"\bsignatures?\b")

# ---------------------------------------------------------------------------
# Sinais que caracterizam um conjunto de demonstrações contábeis
# ---------------------------------------------------------------------------
SINAIS = {
    "balanco":    re.compile(r"balance\s+sheets?|statements?\s+of\s+financial\s+position"),
    "notas":      re.compile(r"notes\s+to\s+.{0,60}?financial\s+statements|"
                             r"notes\s+to\s+the\s+accounts"),
    "resultado":  re.compile(r"statements?\s+of\s+((net|total|consolidated|combined)\s+){0,2}"
                             r"(operations|income|comprehensive|profit|earnings|loss)|"
                             r"income\s+statements?"),
    "fluxo":      re.compile(r"statements?\s+of\s+cash\s+flows?|cash\s+flow\s+statements?"),
    "patrimonio": re.compile(r"statements?\s+of\s+(changes\s+in\s+)?"
                             r"(shareholders|stockholders|owners|equity)"),
    "auditor":    P_AUDITOR,
}
OBRIGATORIOS = ("balanco", "notas")     # sem estes dois, o trecho é texto narrativo, não demonstração
PONTOS_MINIMOS = 4                      # de 6 sinais
PONTOS_BONS    = 5                      # a partir daqui, não vale procurar em outros documentos

# Conteúdo numérico: demonstrações contábeis são densas em números; índices e sumários não.
P_NUMERO = re.compile(r"\d{1,3}(?:,\d{3})+|\d+\.\d{2}\b")
MIN_NUMEROS = 150          # abaixo disso o trecho é índice ou remissão, não demonstração
NUMEROS_BONS = 400         # a partir daqui, não vale procurar em outros documentos
RETENCAO_MINIMA = 0.6      # trecho mais curto que preserve ao menos esta fração dos números

# Declarações de ausência (fundos de securitização, empresas-veículo)
P_OMITIDO = re.compile(r"^.{0,400}?(omitted|not\s+applicable|none\.)", re.S)

MIN_ACEITAVEL = 4000

def pontuar(trecho):
    """(sinais contábeis presentes, quantidade de números). Zero sinais se faltar um obrigatório."""
    presentes = {k for k, p in SINAIS.items() if p.search(trecho)}
    numeros = len(P_NUMERO.findall(trecho))
    if any(o not in presentes for o in OBRIGATORIOS):
        return 0, numeros
    return len(presentes), numeros

def melhor_par(texto, p_ini, p_fim):
    """Par (início, fim) mais distante — evita o sumário, onde os títulos ficam colados."""
    inis = [m.start() for m in p_ini.finditer(texto)]
    fins = [m.start() for m in p_fim.finditer(texto)]
    melhor, tamanho = None, 0
    for i in inis:
        seguintes = [f for f in fins if f > i]
        f = seguintes[0] if seguintes else len(texto)
        if f - i > tamanho:
            melhor, tamanho = (i, f), f - i
    return melhor

def candidatos(texto, formulario):
    """Trechos a testar: pelo item, pelo parecer do auditor e, quando cabe, o documento inteiro."""
    saida = []
    if formulario == "10-K":
        p = melhor_par(texto, P_10K_INI, P_10K_FIM)
        if p:
            saida.append((p[0], p[1], "item8"))
    elif formulario == "20-F":
        for padrao, nome in ((P_20F_INI, "item18"), (P_20F_ALT, "item17")):
            p = melhor_par(texto, padrao, P_20F_FIM)
            if p:
                saida.append((p[0], p[1], nome))
    # Âncoras: o parecer do auditor e o título do balanço. As duas são necessárias porque a
    # ordem varia — em boa parte dos arquivamentos europeus o parecer vem depois das
    # demonstrações, e ancorar só nele deixaria as demonstrações de fora do trecho.
    for padrao, nome in ((P_AUDITOR, "paginas_F"), (SINAIS["balanco"], "balanco")):
        for m in list(padrao.finditer(texto))[:5]:
            ini = max(0, m.start() - 60)       # recua o bastante para não cortar o título ao meio
            fins = [s.start() for s in P_ASSINAT.finditer(texto) if s.start() > ini + MIN_ACEITAVEL]
            saida.append((ini, fins[-1] if fins else len(texto), nome))
    if formulario in ("40-F", "anexo"):
        saida.append((0, len(texto), "documento_integral"))
    return saida

def recortar(html, formulario):
    """Melhor trecho do documento: (html_da_secao, metodo, n_caracteres, pontos, numeros).
    metodo 'sem_demonstracoes' quando o item existe e está declarado como omitido."""
    html = ENTIDADES.sub(" ", html)
    texto, mapa = texto_e_mapa(html)
    if not texto:
        return None, "sem_texto", 0, 0, 0

    validos, omitido = [], False
    for ini, fim, metodo in candidatos(texto, formulario):
        trecho = texto[ini:fim]
        if len(trecho) < MIN_ACEITAVEL:
            if P_OMITIDO.search(trecho):
                omitido = True
            continue
        pontos, numeros = pontuar(trecho)
        if pontos < PONTOS_MINIMOS or numeros < MIN_NUMEROS:
            continue
        validos.append((pontos, numeros, ini, fim, metodo, len(trecho)))

    if not validos:
        return None, ("sem_demonstracoes" if omitido else "nao_localizado"), 0, 0, 0

    # Escolha em duas etapas. Primeiro o maior número de sinais e o maior conteúdo numérico,
    # que é o que separa as demonstrações do índice que as lista. Depois, entre os trechos que
    # preservam a maior parte desse conteúdo, o mais curto — assim o resultado fica nas
    # demonstrações em vez de abarcar o relatório inteiro.
    melhor_pontos = max(v[0] for v in validos)
    fortes = [v for v in validos if v[0] == melhor_pontos]
    teto = max(v[1] for v in fortes)
    proximos = [v for v in fortes if v[1] >= RETENCAO_MINIMA * teto]
    pontos, numeros, ini, fim, metodo, n = min(proximos, key=lambda v: v[5])
    ini_html = mapa[ini]
    fim_html = mapa[fim - 1] if fim - 1 < len(mapa) else len(html)
    return html[ini_html:fim_html], metodo, n, pontos, numeros

MOLDE = """<html><head><meta charset="utf-8"><style>
 @page {{ size: A4; margin: 12mm }}
 body {{ font-family: Georgia, serif; font-size: 9pt; line-height: 1.35 }}
 table {{ border-collapse: collapse; font-size: 7.5pt; width: 100% }}
 td, th {{ padding: 1px 3px; vertical-align: bottom }}
 img {{ display: none }}
 .cabecalho {{ font-size: 8pt; color: #555; border-bottom: 1px solid #999;
               margin-bottom: 8pt; padding-bottom: 4pt }}
</style></head><body>
<div class="cabecalho">{titulo}</div>
{corpo}
</body></html>"""

def texto_de_pdf(bytes_pdf):
    """Texto de um anexo já entregue em PDF, para que ele possa ser avaliado como os demais."""
    with tempfile.NamedTemporaryFile("wb", suffix=".pdf", delete=False) as f:
        f.write(bytes_pdf); tmp = f.name
    try:
        r = subprocess.run(["pdftotext", "-q", tmp, "-"], capture_output=True, text=True,
                           timeout=300)
        return r.stdout
    except Exception:
        return ""
    finally:
        os.unlink(tmp)

def avaliar_pdf(bytes_pdf):
    """(pontos, numeros) de um anexo em PDF."""
    txt = re.sub(r"\s+", " ", texto_de_pdf(bytes_pdf)).lower()
    if len(txt) < MIN_ACEITAVEL:
        return 0, 0
    return pontuar(txt)

def gerar_pdf(html_secao, titulo, destino):
    with tempfile.NamedTemporaryFile("w", suffix=".html", delete=False, encoding="utf-8") as f:
        f.write(MOLDE.format(titulo=titulo, corpo=html_secao)); tmp = f.name
    try:
        subprocess.run(["wkhtmltopdf", "--quiet", "--enable-local-file-access", "--no-images",
                        "--load-error-handling", "ignore", "--load-media-error-handling", "ignore",
                        "--disable-external-links", "--footer-right", "[page]/[topage]",
                        "--footer-font-size", "7", tmp, destino],
                       capture_output=True, timeout=600)
        return os.path.exists(destino) and os.path.getsize(destino) > 1000
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(tmp)

In [ ]:
LOG    = os.path.join(PASTA, "log", "extracao.csv")
FALHAS = os.path.join(PASTA, "log", "falhas.csv")
COLUNAS = ("accession,cik,empresa,ticker,formulario,data,sic,metodo,documento,pontos,numeros,"
           "caracteres,kb_pdf,url")
COL_FALHA = "accession,cik,empresa,formulario,data,sic,motivo,url,documentos_examinados"

_log_lock = threading.Lock()

def registrar(caminho, linha, cabecalho):
    with _log_lock:
        novo = not os.path.exists(caminho)
        with open(caminho, "a", encoding="utf-8") as f:
            if novo:
                f.write(cabecalho + "\n")
            f.write(linha + "\n")

def concluidos():
    feitos = set()
    for c in (LOG, FALHAS):
        if os.path.exists(c):
            feitos |= set(pd.read_csv(c)["accession"].astype(str))
    return feitos

def limpo(s):
    return str(s).replace(",", " ").replace("\n", " ")[:80]

def extrair(linha):
    """Percorre os documentos do protocolo e devolve o trecho de melhor conteúdo contábil.
    (conteudo, metodo, n, pontos, numeros, url, documento, sic, examinados)"""
    url, nome, sic = documento_principal(linha)
    melhor, omitido, examinados = None, False, []

    def considerar(bruto, u, doc, forma):
        """Avalia um documento. bruto em texto (HTML) ou bytes (PDF já pronto)."""
        nonlocal melhor, omitido
        examinados.append(doc)
        if isinstance(bruto, bytes):                     # anexo entregue em PDF
            pontos, numeros = avaliar_pdf(bruto)
            if pontos >= PONTOS_MINIMOS and numeros >= MIN_NUMEROS:
                chave = (pontos, numeros)
                if melhor is None or chave > (melhor[3], melhor[4]):
                    melhor = (bruto, "pdf_original", 0, pontos, numeros, u, doc)
                return pontos, numeros
            return 0, 0
        secao, metodo, n, pontos, numeros = recortar(bruto, forma)
        if metodo == "sem_demonstracoes":
            omitido = True
        if secao:
            chave = (pontos, numeros)
            if melhor is None or chave > (melhor[3], melhor[4]):
                melhor = (secao, metodo, n, pontos, numeros, u, doc)
            return pontos, numeros
        return 0, 0

    def bom(par):
        return par[0] >= PONTOS_BONS and par[1] >= NUMEROS_BONS

    if linha["formulario"] != "40-F" and url:
        bruto = baixar(url)
        if bruto and bom(considerar(bruto, url, nome, linha["formulario"])):
            return melhor + (sic, examinados)
        if omitido:                       # item existe e está declarado como omitido
            return (None, "sem_demonstracoes", 0, 0, 0, url, nome, sic, examinados)

    # Demais documentos do protocolo: anexos do 40-F, demonstrações em arquivo separado
    for outro in arquivos_do_protocolo(linha["url_pasta"], excluir=(nome,) if nome else ())[:12]:
        u = linha["url_pasta"] + outro
        binario = outro.lower().endswith(".pdf")
        bruto = baixar(u, binario=binario)
        if not bruto:
            continue
        if bom(considerar(bruto, u, outro, "anexo")):
            break

    # Último recurso: a submissão completa, que traz documentos fora da listagem da pasta
    if melhor is None and not omitido:
        for doc, corpo in documentos_da_submissao(linha["url_submissao_completa"]):
            if bom(considerar(corpo, linha["url_submissao_completa"], doc, "anexo")):
                break

    if melhor:
        conteudo, metodo, n, pontos, numeros, u, doc = melhor
        if doc != nome:
            metodo += "_em_anexo"
        return (conteudo, metodo, n, pontos, numeros, u, doc, sic, examinados)
    return (None, "sem_demonstracoes" if omitido else "nao_localizado",
            0, 0, 0, url, nome, sic, examinados)

def processar(linha):
    acc = linha["accession"]
    try:
        secao, metodo, n, pontos, numeros, url, doc, sic, examinados = extrair(linha)
        base_falha = (f"{acc},{linha['cik']},{limpo(linha['empresa'])},{linha['formulario']},"
                      f"{linha['data']},{sic}")
        if not secao:
            registrar(FALHAS, f"{base_falha},{metodo},{url},{' '.join(examinados[:10])}",
                      COL_FALHA)
            return metodo if metodo == "sem_demonstracoes" else "falha"

        pasta_ano = os.path.join(PASTA, "pdf", str(linha["ano"]), linha["formulario"])
        os.makedirs(pasta_ano, exist_ok=True)
        tic = linha.get("ticker") if pd.notna(linha.get("ticker")) else "NA"
        base = f"{linha['cik']}_{tic}_{linha['ano']}_{acc}"
        destino = os.path.join(pasta_ano, base + ".pdf")

        titulo = (f"{limpo(linha['empresa'])} — CIK {linha['cik']} — {linha['formulario']} — "
                  f"protocolo {linha['data']} — accession {acc}")
        if isinstance(secao, bytes):          # anexo já entregue em PDF pela própria empresa
            with open(destino, "wb") as f:
                f.write(secao)
        elif not gerar_pdf(secao, titulo, destino):
            registrar(FALHAS, f"{base_falha},pdf_falhou,{url},", COL_FALHA)
            return "falha"

        if SALVAR_HTML_DA_SECAO and not isinstance(secao, bytes):
            ph = os.path.join(PASTA, "html_secao", str(linha["ano"]))
            os.makedirs(ph, exist_ok=True)
            with open(os.path.join(ph, base + ".html"), "w", encoding="utf-8") as f:
                f.write(secao)

        kb = os.path.getsize(destino) // 1024
        registrar(LOG, f"{acc},{linha['cik']},{limpo(linha['empresa'])},{tic},"
                       f"{linha['formulario']},{linha['data']},{sic},{metodo},{doc},{pontos},"
                       f"{numeros},{n},{kb},{url}", COLUNAS)
        return "ok"
    except Exception as e:
        registrar(FALHAS, f"{acc},{linha['cik']},{limpo(linha.get('empresa'))},"
                          f"{linha['formulario']},{linha['data']},,{type(e).__name__},,",
                  COL_FALHA)
        return "erro"

In [ ]:
# Detector de notas mais abrangente. O rótulo literal "notes to ... financial statements"
# não cobre todos os arquivamentos: a Texas Instruments, por exemplo, apresenta as notas
# numeradas sem esse título, e o critério anterior reprovava o trecho inteiro por isso.
P_NOTAS_ALT = re.compile(r"summary\s+of\s+significant\s+accounting\s+policies"
                         r"|significant\s+accounting\s+policies"
                         r"|basis\s+of\s+(presentation|preparation)")
P_NOTA_NUM = re.compile(r"\bnote\s+\d{1,2}\b")

SINAIS["notas"] = re.compile(r"notes?\s+to\s+(the\s+)?[a-z\s\-]{0,40}?financial\s+statements"
                             r"|notes\s+to\s+the\s+accounts")

# Contador que reconhece valores em milhões com uma decimal e negativos entre parênteses
P_NUMERO = re.compile(r"\d{1,3}(?:,\d{3})+|\d+\.\d{1,2}\b|\(\d{2,}\)")

def pontuar(trecho):
    """(sinais contábeis presentes, quantidade de números). Zero se faltar um obrigatório."""
    presentes = {k for k, p in SINAIS.items() if p.search(trecho)}
    if "notas" not in presentes:
        if P_NOTAS_ALT.search(trecho) or len(P_NOTA_NUM.findall(trecho)) >= 4:
            presentes.add("notas")
    numeros = len(P_NUMERO.findall(trecho))
    if any(o not in presentes for o in OBRIGATORIOS):
        return 0, numeros
    return len(presentes), numeros

# Critérios normais
PONTOS_MINIMOS  = 4
MIN_NUMEROS     = 150
MIN_ACEITAVEL   = 4000
NUMEROS_BONS    = 400
RETENCAO_MINIMA = 0.6

print("detectores atualizados")

## 3. Restaurar o registro de falhas

Os relatórios sem demonstrações saíram do `falhas.csv` quando ele foi renomeado no início de cada
reprocessamento. Esta célula devolve essas linhas ao arquivo.

In [ ]:
anterior = pd.read_csv(FALHAS + ".antes_notas", low_memory=False)
atual = (pd.read_csv(FALHAS, low_memory=False) if os.path.exists(FALHAS)
         else pd.DataFrame(columns=anterior.columns))

resgate = anterior[anterior["motivo"] == "sem_demonstracoes"]
juntos = pd.concat([atual, resgate]).drop_duplicates("accession")
juntos.to_csv(FALHAS, index=False)

print(juntos["motivo"].value_counts().to_string())

## 4. Conferência dos recuperados na rodada das notas

São os 106 obtidos com critérios normais. Interessa ver se empresas grandes, como Texas
Instruments e McKesson, saíram com conteúdo compatível com o porte.

In [ ]:
alvo2 = set(anterior[anterior["motivo"] == "nao_localizado"]["accession"].astype(str))
lg = pd.read_csv(LOG, low_memory=False)
novos = lg[lg["accession"].astype(str).isin(alvo2)]

print("recuperados:", len(novos))
print(novos["metodo"].value_counts().to_string())
print("\nconteúdo numérico:")
print(novos["numeros"].describe()[["min", "25%", "50%", "max"]].to_string())
print("\nprimeiras empresas:")
cols = ["empresa", "formulario", "metodo", "pontos", "numeros", "kb_pdf"]
print(novos[cols].head(12).to_string(index=False))

## 5. Reavaliação dos 271

Verifica se os documentos recuperados com critério afrouxado passam agora pelos critérios
normais. Nenhum PDF é refeito — o conteúdo é o mesmo, muda apenas a classificação. Cerca de
cinco minutos.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

primeiro = pd.read_csv(FALHAS + ".anterior", low_memory=False)
antes  = set(primeiro[primeiro["motivo"] == "nao_localizado"]["accession"].astype(str))
depois = set(anterior[anterior["motivo"] == "nao_localizado"]["accession"].astype(str))
ampliado = antes - depois

trab = pd.read_csv(os.path.join(PASTA, "indice", "trabalho.csv"), low_memory=False)
casos = trab[trab["accession"].astype(str).isin(ampliado)].to_dict("records")
print("a reavaliar:", len(casos))

def reavaliar(linha):
    url, doc, sic = documento_principal(linha)
    bruto = baixar(url) if url else None
    if not bruto:
        return (str(linha["accession"]), False)
    secao, metodo, n, pontos, numeros = recortar(bruto, linha["formulario"])
    return (str(linha["accession"]), secao is not None)

with ThreadPoolExecutor(max_workers=N_THREADS) as pool:
    resultados = list(pool.map(reavaliar, casos))

passaram = {a for a, ok in resultados if ok}
ainda = ampliado - passaram
print("passam no critério normal:", len(passaram), "de", len(resultados))
print("seguem com ressalva:", len(ainda))

pd.DataFrame({"accession": sorted(ainda)}).to_csv(
    os.path.join(PASTA, "indice", "ainda_ampliado.csv"), index=False)

## 6. Tabela final

Regrava o `painel_final.csv` e marca com ressalva apenas os que continuam dependendo de critério
afrouxado.

In [ ]:
import glob

casa = pd.read_csv(os.path.join(PASTA, "indice", "painel_casado.csv"), low_memory=False)
lg = pd.read_csv(LOG, low_memory=False)
fa = pd.read_csv(FALHAS, low_memory=False)

arquivos = glob.glob(os.path.join(PASTA, "pdf", "**", "*.pdf"), recursive=True)
por_acc = {}
for p in arquivos:
    acc = os.path.basename(p).rsplit(".", 1)[0].split("_")[-1]
    por_acc[acc] = os.path.relpath(p, PASTA)
print("PDFs indexados:", len(por_acc))

m = casa.copy()
m["accession"] = m["accession"].astype(str)
m["arquivo"] = m["accession"].map(por_acc)
m["cik_do_arquivo"] = m["arquivo"].str.extract(r"/(\d+)_[^/]*\.pdf$")

falhos = dict(zip(fa["accession"].astype(str), fa["motivo"]))
feitos = set(lg["accession"].astype(str))

def situacao(r):
    if pd.isna(r["accession"]) or r["accession"] == "nan":
        return "sem registro na SEC"
    if isinstance(r["arquivo"], str):
        return "extraído"
    if r["accession"] in falhos:
        return "falha: " + str(falhos[r["accession"]])
    if r["accession"] in feitos:
        return "extraído (arquivo não localizado)"
    return "pendente"

m["situacao"] = m.apply(situacao, axis=1)

com_ressalva = set(pd.read_csv(
    os.path.join(PASTA, "indice", "ainda_ampliado.csv"))["accession"].astype(str))
marcar = m["accession"].isin(com_ressalva) & (m["situacao"] == "extraído")
m.loc[marcar, "situacao"] = "extraído (critério ampliado)"

cols = ["cik", "conm", "tic", "Year", "datadate", "formulario",
        "accession", "situacao", "arquivo", "cik_do_arquivo"]
final = m[cols].rename(columns={"conm": "empresa", "tic": "ticker", "Year": "ano_fiscal"})
final.to_csv(os.path.join(PASTA, "indice", "painel_final.csv"), index=False)

print()
print(final["situacao"].value_counts().to_string())
compartilhado = (final["cik_do_arquivo"].notna()
                 & (final["cik_do_arquivo"].astype(str) != final["cik"].astype(str)))
print("\ncompartilhados com outra empresa:", int(compartilhado.sum()))

## Encerramento

O `painel_final.csv` é o arquivo que acompanha a entrega: uma linha por empresa-ano, com a
situação, o caminho do PDF e o CIK sob o qual o arquivo foi gravado — que difere do CIK da
empresa quando o relatório é conjunto, caso de holdings de energia e companhias aéreas.

Os arquivos `falhas.csv.anterior` e `falhas.csv.antes_notas` documentam os dois reprocessamentos
e devem ser preservados: são eles que permitem reconstruir quais documentos passaram por qual
tratamento, caso isso precise ser descrito no método.